# Imports

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

listings_rec = pd.read_csv('listings_rec.csv')
print(f'Loaded {len(listings_clean)} listings')
print(listings_clean[['price_log', 'has_reviews', 'review_scores_rating', 'minimum_nights']].head())

Loaded 36613 listings
   price_log  has_reviews  review_scores_rating  minimum_nights
0   4.454347            1                  4.89              30
1   5.484797            1                  4.68              30
2   5.379897            1                  4.75              30
3   4.574711            1                  4.58              30
4   4.094345            1                  5.00              30


In [14]:
# define feature columns
feature_cols = ['price_log', 'accommodates', 'review_scores_rating',
                'has_reviews', 'minimum_nights', 'room_type',
                'neighbourhood_group_cleansed']

listings_encoded = pd.get_dummies(
    listings_clean[feature_cols + ['id']],
    columns=['room_type', 'neighbourhood_group_cleansed'],
    drop_first=False
)

# normalise
scaler = StandardScaler()
feature_matrix = scaler.fit_transform(listings_encoded.drop('id', axis=1))

print(f'Feature matrix shape: {feature_matrix.shape}')
print(f'Columns: {list(listings_encoded.drop("id", axis=1).columns)}')

feature_columns = list(listings_encoded.drop('id', axis=1).columns)
print(feature_columns)

Feature matrix shape: (36613, 14)
Columns: ['price_log', 'accommodates', 'review_scores_rating', 'has_reviews', 'minimum_nights', 'room_type_Entire home/apt', 'room_type_Hotel room', 'room_type_Private room', 'room_type_Shared room', 'neighbourhood_group_cleansed_Bronx', 'neighbourhood_group_cleansed_Brooklyn', 'neighbourhood_group_cleansed_Manhattan', 'neighbourhood_group_cleansed_Queens', 'neighbourhood_group_cleansed_Staten Island']
['price_log', 'accommodates', 'review_scores_rating', 'has_reviews', 'minimum_nights', 'room_type_Entire home/apt', 'room_type_Hotel room', 'room_type_Private room', 'room_type_Shared room', 'neighbourhood_group_cleansed_Bronx', 'neighbourhood_group_cleansed_Brooklyn', 'neighbourhood_group_cleansed_Manhattan', 'neighbourhood_group_cleansed_Queens', 'neighbourhood_group_cleansed_Staten Island']


# Recommender

In [22]:
def recommend_listings(user_prefs, feature_matrix, listings_rec, 
                       feature_columns, scaler, top_n=5):
   
    query = pd.DataFrame(0, index=[0], columns=feature_columns)

    for key, value in user_prefs.items():
        if key in query.columns:
            query[key] = value
    
    query_scaled = scaler.transform(query)

    similarities = cosine_similarity(query_scaled, feature_matrix)[0]
    
    top_indices = similarities.argsort()[::-1][:top_n]
    
    results = listings_rec.iloc[top_indices][[
        'id', 'price', 'room_type',
        'neighbourhood_group_cleansed',
        'review_scores_rating',
        'accommodates',
        'minimum_nights'
    ]].copy()
    results['similarity_score'] = similarities[top_indices].round(3)
    
    return results.reset_index(drop=True)

print('Recommender is ready')

Recommender is ready


# Sample Users Test

In [19]:
# synthetic user 1 -- budget private room in Brooklyn
user_1 = {
    'price_log': np.log1p(80),        # budget ~80 USD/night
    'accommodates': 2,                  # 2 guests
    'review_scores_rating': 4.8,        # high review score
    'has_reviews': 1,                   # wants reviewed listings
    'minimum_nights': 2,                # short stay
    'room_type_Private room': 1,        # private room
    'neighbourhood_group_cleansed_Brooklyn': 1  # Brooklyn
}

results_1 = recommend_listings(user_1, feature_matrix, listings_rec,
                                feature_columns, scaler, top_n=5)
print('=== Budget Private Room in Brooklyn ===')
print(results_1)

=== Budget Private Room in Brooklyn ===
                    id  price     room_type neighbourhood_group_cleansed  \
0             23040115   85.0  Private room                     Brooklyn   
1              2385779   85.0  Private room                     Brooklyn   
2  1129406858712912754   85.0  Private room                     Brooklyn   
3             49923953   86.0  Private room                     Brooklyn   
4   717557610336262134   86.0  Private room                     Brooklyn   

   review_scores_rating  accommodates  minimum_nights  similarity_score  
0                  4.80             2               2             1.000  
1                  4.80             2               3             0.999  
2                  4.80             2               1             0.999  
3                  4.79             2               3             0.999  
4                  4.81             2               1             0.999  


In [25]:
# user 2 -- luxury entire home in Manhattan
user_2 = {
    'price_log': np.log1p(400),
    'accommodates': 4,
    'review_scores_rating': 4.9,
    'has_reviews': 1,
    'minimum_nights': 3,
    'room_type_Entire home/apt': 1,
    'neighbourhood_group_cleansed_Manhattan': 1
}

print('=== Luxury Entire Home in Manhattan ===')
print(recommend_listings(user_2, feature_matrix, listings_rec,
                          feature_columns, scaler, top_n=5))

=== Luxury Entire Home in Manhattan ===
                    id  price        room_type neighbourhood_group_cleansed  \
0             38850758  393.0  Entire home/apt                    Manhattan   
1   762429245666943839  410.0  Entire home/apt                    Manhattan   
2   990420260648778710  422.0  Entire home/apt                    Manhattan   
3  1127492187924840140  409.0  Entire home/apt                    Manhattan   
4  1131888962475292890  395.0  Entire home/apt                    Manhattan   

   review_scores_rating  accommodates  minimum_nights  similarity_score  
0                  4.90             4               2             1.000  
1                  4.92             4               1             1.000  
2                  4.93             4               3             0.999  
3                  4.87             4               5             0.999  
4                  4.94             4               1             0.999  


In [26]:
# user 3 -- budget shared room in Queens
user_3 = {
    'price_log': np.log1p(50),
    'accommodates': 1,
    'review_scores_rating': 4.5,
    'has_reviews': 0,
    'minimum_nights': 1,
    'room_type_Shared room': 1,
    'neighbourhood_group_cleansed_Queens': 1
}

print()
print('=== Budget Shared Room in Queens ===')
print(recommend_listings(user_3, feature_matrix, listings_rec,
                          feature_columns, scaler, top_n=5))


=== Budget Shared Room in Queens ===
                    id  price    room_type neighbourhood_group_cleansed  \
0             13697001   56.0  Shared room                       Queens   
1             23169146   56.0  Shared room                       Queens   
2              9944320   56.0  Shared room                       Queens   
3             23747870   56.0  Shared room                       Queens   
4  1373609435734465471   33.0  Shared room                       Queens   

   review_scores_rating  accommodates  minimum_nights  similarity_score  
0                  4.86             1              30             0.996  
1                  4.86             1              30             0.996  
2                  4.86             1              30             0.996  
3                  4.86             1              30             0.996  
4                  4.86             1              30             0.996  


## Gradient Descent Feature Weighting

The cosine similarity recommender treats all features equally after scaling.
But based on Project 1, the features differ significantly in importance --
review_scores_rating had a t-statistic of 54.5 vs price at 0.18.

Gradient descent targest this by learning the optimal feature weights through minimising the difference
between predicted and actual occupancy.


In [34]:
def gradient_descent(X, y, learning_rate=0.01, epochs=1000):
    
    # initialise weights to zero
    weights = np.zeros(X.shape[1])
    loss_history = []
    
    for epoch in range(epochs):
        # forward pass -- predicted occupancy
        y_pred = X @ weights
        
        # compute loss -- mean squared error
        loss = np.mean((y - y_pred) ** 2)
        loss_history.append(loss)
        
        # compute gradient
        gradient = -2/len(y) * X.T @ (y - y_pred)
        
        # update weights
        weights = weights - learning_rate * gradient
        
        # print progress every 200 epochs
        if epoch % 200 == 0:
            print(f'Epoch {epoch}: loss = {loss:.6f}')
    
    return weights, loss_history

# filter and normalise BEFORE calling gradient_descent
active_mask = listings_rec['estimated_occupancy_l365d'] > 0
X_active = feature_matrix[active_mask.values]
y_active = listings_rec.loc[active_mask, 'estimated_occupancy_l365d'].values
y_norm = (y_active - y_active.min()) / (y_active.max() - y_active.min())

# now call the function with clean inputs
weights, loss_history = gradient_descent(X_active, y_norm, 
                                          learning_rate=0.01, epochs=2000)

Epoch 0: loss = 0.462941
Epoch 200: loss = 0.118459
Epoch 400: loss = 0.111231
Epoch 600: loss = 0.110471
Epoch 800: loss = 0.110309
Epoch 1000: loss = 0.110272
Epoch 1200: loss = 0.110263
Epoch 1400: loss = 0.110261
Epoch 1600: loss = 0.110261
Epoch 1800: loss = 0.110261
